# Computational cost

The notebooks `01_baselines`, `02_sliding_window` and `03_oja` each report the runtime of their own methods. Those timings come from different runs, made at different times and under a different load of the computer, so they can be compared within a notebook but not across notebooks. This notebook times all the methods in the same run, on the same stream.

### Organization

1. Setup: one stationary stream and the three memory levels of the previous notebooks
2. Runtime of every method
3. What the timings show

In [1]:
from pathlib import Path
import sys
import platform

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'methods.py').is_file() and (p / 'data_generators.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from online_pca_thesis or its notebooks folder.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

import plots as ut
from data_generators import generate_stationary
from methods import (cumulative_pca, forgetting_pca, incremental_pca, rank1_svd_pca, sliding_window_pca,
                     sliding_window_cov_pca, sliding_window_svd_pca, tw_api_pca, oja_pca)
from benchmark import best_time, write_rows

## 1. Setup

The runtime of these methods does not depend on the data regime (every observation costs the same, whatever the direction does), so one stationary stream is enough. It has the settings of the previous notebooks: $2000$ observations, $\lambda_1=5$, $\lambda_2=0.5$, seed $0$.

The methods are compared at the three memory levels of the previous notebooks: forgetting gain $\alpha\in\{0.005,0.02,0.1\}$, the window $w=2/\alpha-1\in\{399,99,19\}$ with the same stationary noise (`02_sliding_window`), and the Oja gain $\alpha/(\lambda_1-\lambda_2)$ (`03_oja`). Cumulative and incremental PCA have no parameter: for them the three columns are three repeated measurements of the same computation, and show how much the timings vary.

In [ ]:
n_samples = 2000
lam1 = 5.0
lam2 = 0.5
seed = 0
gains = [0.005, 0.02, 0.1]                              # forgetting gains, from long to short memory
windows = [round(2 / alpha - 1) for alpha in gains]     # equal-noise windows: 399, 99, 19
oja_gains = [alpha / (lam1 - lam2) for alpha in gains]  # equal-noise Oja gains
rounds = 5                                              # timing rounds; the shortest time is kept

X = generate_stationary(seed, n_samples=n_samples, lam1=lam1, lam2=lam2)[0]
print('windows:', windows, ' Oja gains:', [f'{a:.5f}' for a in oja_gains])

windows: [399, 99, 19]  Oja gains: ['0.00111', '0.00444', '0.02222']
Python 3.12.2 | NumPy 1.26.4 | Intel64 Family 6 Model 154 Stepping 3, GenuineIntel


## 2. Runtime of every method

Each method processes the whole stream; the time is divided by the number of observations. The methods are timed in turn within each round, and for each method the shortest of the `rounds` times is kept: the minimum is the measurement least disturbed by other processes, and alternating the methods spreads any change in the load of the computer over all of them.

In [3]:
methods = [
    ('Cumulative PCA', [lambda: cumulative_pca(X)] * 3),
    ('Forgetting PCA', [lambda a=a: forgetting_pca(X, a) for a in gains]),
    ('Incremental PCA (rank-1 SVD)', [lambda: incremental_pca(X)] * 3),
    ('Rank-1 SVD with forgetting', [lambda a=a: rank1_svd_pca(X, a) for a in gains]),
    ('Sliding window, naive recomputation', [lambda w=w: sliding_window_pca(X, w) for w in windows]),
    ('Sliding window, covariance update/downdate', [lambda w=w: sliding_window_cov_pca(X, w) for w in windows]),
    ('Sliding window, rank-1 SVD update/downdate', [lambda w=w: sliding_window_svd_pca(X, w) for w in windows]),
    ('Sliding window, TW-API', [lambda w=w: tw_api_pca(X, w) for w in windows]),
    ('Oja PCA', [lambda a=a: oja_pca(X, a) for a in oja_gains]),
]

times = {(name, level): np.inf for name, runs in methods for level in range(3)}
for _ in range(rounds):
    for name, runs in methods:
        for level, run in enumerate(runs):
            times[name, level] = min(times[name, level], best_time(run, repeats=1))

per_step = {key: value / n_samples * 1e6 for key, value in times.items()}   # microseconds per observation
reference = np.mean([per_step['Forgetting PCA', level] for level in range(3)])
rows = []
for name, _ in methods:
    values = [per_step[name, level] for level in range(3)]
    rows.append([name] + [f'{v:.1f}' for v in values] + [f'{np.mean(values) / reference:.2f}'])
ut.show_table(['Method', 'α=0.005 / w=399 (µs)', 'α=0.02 / w=99 (µs)', 'α=0.1 / w=19 (µs)',
               'Relative to forgetting PCA'], rows)

results_dir = PROJECT_ROOT / 'results' / 'computational_cost'
results_dir.mkdir(parents=True, exist_ok=True)
write_rows(results_dir / 'runtime_per_observation.csv',
           [{'method': name, 'memory_level': level, 'forgetting_alpha': gains[level], 'window_size': windows[level],
             'oja_alpha': oja_gains[level], 'microseconds_per_observation': per_step[name, level],
             'n_samples': n_samples, 'rounds': rounds, 'numpy_version': np.__version__,
             'python_version': platform.python_version()}
            for name, _ in methods for level in range(3)])

| Method | α=0.005 / w=399 (µs) | α=0.02 / w=99 (µs) | α=0.1 / w=19 (µs) | Relative to forgetting PCA |
| --- | --- | --- | --- | --- |
| Cumulative PCA | 24.6 | 23.5 | 23.9 | 1.00 |
| Forgetting PCA | 24.3 | 23.8 | 24.3 | 1.00 |
| Incremental PCA (rank-1 SVD) | 25.3 | 24.8 | 25.3 | 1.04 |
| Rank-1 SVD with forgetting | 25.8 | 25.1 | 25.0 | 1.05 |
| Sliding window, naive recomputation | 24.1 | 23.4 | 22.7 | 0.97 |
| Sliding window, covariance update/downdate | 26.9 | 27.4 | 26.8 | 1.12 |
| Sliding window, rank-1 SVD update/downdate | 41.0 | 44.7 | 45.5 | 1.81 |
| Sliding window, TW-API | 10.7 | 11.0 | 11.2 | 0.45 |
| Oja PCA | 6.6 | 6.3 | 6.4 | 0.27 |


Saved: C:\Users\mesie\OneDrive\Desktop\TESI\online_pca_thesis\results\computational_cost\runtime_per_observation.csv


## 3. Results

| Method | Main operation at each observation | Memory kept |
|---|---|---|
| Cumulative, forgetting PCA | update of the $2\times2$ covariance, eigendecomposition | the covariance |
| Incremental PCA, rank-1 SVD with forgetting | rank-one SVD update | one direction and its singular value |
| Window, naive recomputation | covariance of the whole window, eigendecomposition | the $w$ observations of the window |
| Window, covariance update/downdate | covariance update and downdate, eigendecomposition | the covariance and the window |
| Window, rank-1 SVD update/downdate | two rank-one SVD modifications | one direction, its singular value and the window |
| Window, TW-API | a few vector products, no decomposition | the direction and the window |
| Oja PCA | a few vector products and a normalization, no decomposition | the direction |

- **Oja is the fastest method:** about $6.5$ µs per observation, about a quarter of the time of forgetting PCA ($24$ µs) and less than the fastest window method (TW-API, $11$ µs). The other methods cost about the same as forgetting PCA, except the rank-1 SVD window ($41$–$46$ µs), which makes two SVD modifications per observation.
- **In two dimensions the runtime is set by whether a step calls a decomposition, not by how much data it uses.** Every method that computes an eigendecomposition or an SVD at each observation costs about $23$–$27$ µs, whatever it does around it: the naive window, which recomputes the covariance of all $w$ observations at each step, costs the same at $w=19$ and $w=399$, because that product of at most $399\times2$ numbers is a single fast NumPy operation, small next to the eigendecomposition and the Python loop. Oja and TW-API never call a decomposition and are the only methods clearly below.
- **The runtime does not depend on the memory level.** For each method the three columns differ by a few percent, about as much as the repeated measurements of cumulative and incremental PCA: the gain or the window length changes the result, not the work per observation.
- **These numbers are for this implementation in two dimensions.** The methods are written as Python loops over the observations, and the absolute values depend on the computer. In $d$ dimensions the difference would grow: Oja costs $O(d)$ operations per observation, a covariance method $O(d^2)$ to update the covariance and $O(d^3)$ for a full eigendecomposition. The memory column matters there as well: in two dimensions a $2\times2$ covariance and a direction are both negligible.